# Step 1: lock in the RAMP baseline (official code), resumable via Hugging Face

**One-time setup**
1. Create a Hugging Face *write* token at https://huggingface.co/settings/tokens.
2. In Kaggle: *Add-ons → Secrets → Add secret*, name it `HF_TOKEN`, and enable it for this notebook.
3. Set `HF_REPO` below to `<your-hf-username>/aat-checkpoints`. It is created **private** on first push.

**Each session:** Accelerator *GPU T4 x2*, Internet *on*, then **Save Version → Save & Run All**. The run continues in the background. Each run stops cleanly after 11 h and pushes its checkpoint. Run the notebook again and it resumes from the Hub. Runs that are already finished are skipped automatically.

In [1]:
import os
os.environ['HF_REPO'] = 'matokebryan/aat-checkpoints'   # <- edit once
os.environ['TIME_BUDGET_H'] = '11'
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
%cd /kaggle/working
!git clone -q -b matt/funny-planck-3j7md5 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT
!pip install -q pyyaml
!bash scripts/ramp_official.sh setup

/kaggle/working
/kaggle/working/AAT
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/local/lib/python3.12/dist-packages/robustbench/__init__.py", line 1, in <module>
    from .data import load_cifar10
  File "/usr/local/lib/python3.12/dist-packages/robustbench/data.py", line 15, in <module>
    from robustbench.model_zoo import model_dicts as all_models
  File "/usr/local/lib/python3.12/dist-packages/robustbench/model_zoo/__init__.py", line 1, in <module>
    from .models import model_dicts
  File "/usr/local/lib/python3.12/dist-packages/robustbench/model_zoo/models.py", line 4, in <module>
    from robustbench.model_zoo.cifar10 import cifar_10_models
  File "/usr/local/lib/python3.12/dist-packages/robustbench/model_zoo/cifar10.py", line 20, in <module>
    from robustbench.model_zoo.architectures.sodef_layers import rebuffi_sodef
  File "/usr/local/lib/python3.12/dis

## Sanity check: the official pretrained ℓ∞ model (expect ≈ 83.7 / 48.1 / 59.8 / 7.7 / 38.5)

In [2]:
if not os.path.exists('runs_official/pretr_linf/eval_autoattack.json'):
    !bash scripts/ramp_official.sh pretr 0

100%|█████████████████████████████████████████| 170M/170M [25:55<00:00, 110kB/s]
pretr_linf: linf eps=0.03137 robust=0.4810
pretr_linf: l2 eps=0.5 robust=0.5980
pretr_linf: l1 eps=12 robust=0.0700
pretr_linf: unseen linf eps=0.01569 robust=0.6970
pretr_linf: unseen linf eps=0.04706 robust=0.3000
pretr_linf: unseen linf eps=0.06275 robust=0.1470
pretr_linf: unseen l2 eps=0.25 robust=0.7470
pretr_linf: unseen l2 eps=1 robust=0.2890
pretr_linf: unseen l2 eps=1.5 robust=0.0750
pretr_linf: unseen l1 eps=6 robust=0.3120
pretr_linf: unseen l1 eps=18 robust=0.0130
pretr_linf: unseen l1 eps=24 robust=0.0030
{"name": "pretr_linf", "clean": 0.8370000123977661, "robust": {"linf": 0.48100000619888306, "l2": 0.5979999899864197, "l1": 0.07000000029802322}, "union": 0.07000000029802322, "union_worst_class": 0.0}


## A. Thesis baseline: RAMP from scratch (λ=5, 80 epochs, GP), the setting of thesis Table 7.1 / paper Table 3
Two seeds run in parallel, one per GPU. Re-run the notebook each session until both print `done already`. Then change the seeds to `"2" "3"`, then `"4"`.

In [3]:
!(bash scripts/ramp_official.sh train ramp_scratch "0" 0 & bash scripts/ramp_official.sh train ramp_scratch "1" 1 & wait)
!grep -h "epoch\]" external/ramp/trained_models/ramp_scratch_l5_s*/log_train.txt | tail -n 4   # epoch time -> sessions needed

>> ramp_scratch_l5_s1 on GPU 1
>> ramp_scratch_l5_s0 on GPU 0
[ramp_scratch_l5_s1] Traceback (most recent call last):
[ramp_scratch_l5_s1]   File "/kaggle/working/AAT/external/ramp/RAMP.py", line 19, in <module>
[ramp_scratch_l5_s1]     import robustbench as rb
[ramp_scratch_l5_s1]   File "/usr/local/lib/python3.12/dist-packages/robustbench/__init__.py", line 1, in <module>
[ramp_scratch_l5_s1]     from .data import load_cifar10
[ramp_scratch_l5_s1]   File "/usr/local/lib/python3.12/dist-packages/robustbench/data.py", line 15, in <module>
[ramp_scratch_l5_s1]     from robustbench.model_zoo import model_dicts as all_models
[ramp_scratch_l5_s1]   File "/usr/local/lib/python3.12/dist-packages/robustbench/model_zoo/__init__.py", line 1, in <module>
[ramp_scratch_l5_s1]     from .models import model_dicts
[ramp_scratch_l5_s1]   File "/usr/local/lib/python3.12/dist-packages/robustbench/model_zoo/models.py", line 4, in <module>
[ramp_scratch_l5_s1]     from robustbench.model_zoo.cifar10 impor

In [4]:
# once the seeds are finished
!bash scripts/ramp_official.sh eval ramp_scratch "0 1" 0
!python scripts/compare_targets.py --runs runs_official --table thesis_table7_1 --map ramp_scratch_official=ramp
!python scripts/compare_targets.py --runs runs_official --table scratch_table3 --map ramp_scratch_official=ramp_l5

[hub] nothing to pull for ramp_official/ramp_scratch_l5_s0/ep_80_0.pth (RemoteEntryNotFoundError)
| run | n | metric | ours | paper | Δ | verdict |
|---|---|---|---|---|---|---|
| ramp_scratch_official | 0 | – | not evaluated | | | |

overall: NOT EVALUATED
| run | n | metric | ours | paper | Δ | verdict |
|---|---|---|---|---|---|---|
| ramp_scratch_official | 0 | – | not evaluated | | | |

overall: NOT EVALUATED


## B. Cheap cross-check: RAMP fine-tuning (paper Table 24), 5 seeds, about 1 GPU-hour each. Run it on the GPU time left over in a session

In [5]:
!(bash scripts/ramp_official.sh train ramp "0 2 4" 0 & bash scripts/ramp_official.sh train ramp "1 3" 1 & wait)
!(bash scripts/ramp_official.sh eval ramp "0 2 4" 0 & bash scripts/ramp_official.sh eval ramp "1 3" 1 & wait)
!python scripts/compare_targets.py --runs runs_official --map ramp_official=ramp_l1.5 pretr_linf=pretr_linf

>> ramp_ft_s1 on GPU 1
>> ramp_ft_s0 on GPU 0
[ramp_ft_s1] Traceback (most recent call last):
[ramp_ft_s1]   File "/kaggle/working/AAT/external/ramp/RAMP.py", line 19, in <module>
[ramp_ft_s1]     import robustbench as rb
[ramp_ft_s1]   File "/usr/local/lib/python3.12/dist-packages/robustbench/__init__.py", line 1, in <module>
[ramp_ft_s1]     from .data import load_cifar10
[ramp_ft_s1]   File "/usr/local/lib/python3.12/dist-packages/robustbench/data.py", line 15, in <module>
[ramp_ft_s1]     from robustbench.model_zoo import model_dicts as all_models
[ramp_ft_s1]   File "/usr/local/lib/python3.12/dist-packages/robustbench/model_zoo/__init__.py", line 1, in <module>
[ramp_ft_s1]     from .models import model_dicts
[ramp_ft_s1]   File "/usr/local/lib/python3.12/dist-packages/robustbench/model_zoo/models.py", line 4, in <module>
[ramp_ft_s1]     from robustbench.model_zoo.cifar10 import cifar_10_models
[ramp_ft_s1]   File "/usr/local/lib/python3.12/dist-packages/robustbench/model_zoo/cif